# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikj5453/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring** · iterate month: `2026-03` · sealed test month: `2026-06` (never touched)

## 0) Setup (not one of the three verification queries)
Token comes from Colab Secrets (`HF_TOKEN`) — never pasted in a cell.

In [ ]:
%pip -q install duckdb scikit-learn pandas
import duckdb, pandas as pd, numpy as np
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    HF_TOKEN = getpass("HF read token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

MONTH = "2026-03"
# Folder shape: check the dataset's Files tab. If this 404s, try the commented alternative.
FACT = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet')"
# FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"  # then add: WHERE month = '2026-03'
con.sql(f"DESCRIBE SELECT * FROM {FACT}").df()

**Column names.** Cell above prints the real schema. If any name below differs, change it once here and nowhere else.

In [ ]:
C = dict(
    date   = "report_date",
    client = "client_id",
    content= "content_id",
    impr   = "gsc_impressions",
    clicks = "gsc_clicks",
    pos    = "gsc_avg_position",
    ga4_ok = "ga4_data_available",
)

## 1. Unit of analysis + time window
*The contract in five plain-words answers.*
1. **One row means:** one page (content item) on one day for one client — a `report_date × client × content` row of search performance. For modeling I roll it up to **one row per page for the month**.
2. **Tables:** `fact_content_daily_performance` only, partition `month=2026-03`. (No `fact_content_query_90d` — its 90-day window overlaps the final months and would need window alignment first.)
3. **Time window:** features from **1–15 March 2026**; the decision moment is end of day 15 March. The outcome window is **16–31 March**. June 2026 is sealed and unused.
4. **What I rank:** pages most likely to be **declining** — proxy label `declining = 1` when impressions in 16–31 Mar fell more than 20% vs 1–15 Mar (only pages with ≥ 100 impressions in 1–15 Mar, so the ratio means something). This is a proxy for "review for refresh", not proof a refresh would pay off.
5. **Deliberately excluded:** every column from 16–31 March (the label window), and all GA4 columns (zero-filled before `ga4_data_start`, so zeros mean "not measured"). IDs are context only.

## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `log_impr_h1`, `ctr_h1`, `avg_pos_h1`, `active_days_h1`, `wk2_vs_wk1` | all computed from 1–15 March only |
| **Label** | `declining` (from `impr_h2` vs `impr_h1`) | outcome window 16–31 March |
| **Context** | `client_id`, `content_id`, `report_date`, `month` | grouping, joining, client-holdout split — never learned from |
| **Excluded** | every column from 16–31 March; all GA4 columns; `h2_over_h1` | label-window information is the future; GA4 is zero-filled before each client's `ga4_data_start`, so zeros mean "not measured" |

## 3. Verify it with queries (grain, counts, availability)
*Three queries on the mid-panel month `2026-03`.*
**Query 1 — grain.** Zero rows back means one row really is page × client × day.

In [ ]:
q1 = con.sql(f"""
SELECT {C['date']}, {C['client']}, {C['content']}, COUNT(*) AS c
FROM {FACT}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print("duplicate-grain rows:", len(q1))
q1

**Query 2 — row count and date span** of my slice.

In [ ]:
con.sql(f"""
SELECT COUNT(*)                      AS n_rows,
       MIN({C['date']})              AS first_day,
       MAX({C['date']})              AS last_day,
       COUNT(DISTINCT {C['client']}) AS n_clients,
       COUNT(DISTINCT {C['content']}) AS n_pages
FROM {FACT}
""").df()

**Query 3 — availability.** Filter with `IS TRUE`. The `= FALSE` and NULL counts are shown to prove why `IS TRUE` is the safe form (NULL is neither TRUE nor FALSE).

In [ ]:
con.sql(f"""
SELECT COUNT(*)                                              AS total_rows,
       COUNT(*) FILTER (WHERE {C['ga4_ok']} IS TRUE)         AS ga4_available_is_true,
       COUNT(*) FILTER (WHERE {C['ga4_ok']} = FALSE)         AS flag_eq_false,
       COUNT(*) FILTER (WHERE {C['ga4_ok']} IS NULL)         AS flag_null
FROM {FACT}
""").df()

**Reading the output:** write one sentence here after you run it — how many rows survive `IS TRUE`, and how many NULLs a naive `= FALSE` / `NOT flag` filter would have silently dropped.

### 3b. Five features + the trap
Features use only 1–15 March. Label uses 16–31 March. Pages need ≥ 100 impressions in the feature window.

In [ ]:
D = f"CAST({C['date']} AS DATE)"
frame = con.sql(f"""
WITH p AS (
  SELECT {C['client']} AS client_id, {C['content']} AS content_id,
    SUM({C['impr']})   FILTER (WHERE {D} <= DATE '{MONTH}-15') AS impr_h1,
    SUM({C['clicks']}) FILTER (WHERE {D} <= DATE '{MONTH}-15') AS clicks_h1,
    COUNT(*) FILTER (WHERE {D} <= DATE '{MONTH}-15' AND {C['impr']} > 0) AS active_days_h1,
    AVG({C['pos']}) FILTER (WHERE {D} <= DATE '{MONTH}-15' AND {C['impr']} > 0 AND {C['pos']} > 0) AS avg_pos_h1,
    SUM({C['impr']}) FILTER (WHERE {D} <= DATE '{MONTH}-07') AS impr_wk1,
    SUM({C['impr']}) FILTER (WHERE {D} BETWEEN DATE '{MONTH}-08' AND DATE '{MONTH}-15') AS impr_wk2,
    SUM({C['impr']}) FILTER (WHERE {D} >  DATE '{MONTH}-15') AS impr_h2      -- label window
  FROM {FACT}
  GROUP BY 1, 2
)
SELECT *, impr_h2 * 1.0 / impr_h1 AS h2_over_h1,
       CASE WHEN impr_h2 < 0.8 * impr_h1 THEN 1 ELSE 0 END AS declining
FROM p WHERE impr_h1 >= 100
""").df()

frame["log_impr_h1"]  = np.log1p(frame["impr_h1"])
frame["ctr_h1"]       = 100 * frame["clicks_h1"] / frame["impr_h1"]
frame["wk2_vs_wk1"]   = (frame["impr_wk2"] - frame["impr_wk1"]) / frame["impr_wk1"].replace(0, np.nan)
frame["wk2_vs_wk1"]   = frame["wk2_vs_wk1"].fillna(0)
frame["avg_pos_h1"]   = frame["avg_pos_h1"].fillna(frame["avg_pos_h1"].median())
print(frame.shape, "| positive rate:", round(frame["declining"].mean(), 3))
frame.head()

**Features — "knowable at the decision moment because…"** (decision moment = end of 15 March)

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `log_impr_h1` | it sums impressions from 1–15 March only — all already happened by the 15th |
| 2 | `ctr_h1` | clicks ÷ impressions over 1–15 March, both closed by the 15th |
| 3 | `avg_pos_h1` | mean of the daily average position over active days 1–15 March — daily values are final once the day ends |
| 4 | `active_days_h1` | count of days with impressions in 1–15 March — a history count, nothing after the 15th |
| 5 | `wk2_vs_wk1` | compares 8–15 March to 1–7 March impressions — both weeks end on or before the 15th |

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

FEATS = ["log_impr_h1", "ctr_h1", "avg_pos_h1", "active_days_h1", "wk2_vs_wk1"]

def quick_score(cols):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
    tr, te = next(gss.split(frame, groups=frame["client_id"]))   # client-holdout
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0)
    m.fit(frame.iloc[tr][cols], frame.iloc[tr]["declining"])
    return roc_auc_score(frame.iloc[te]["declining"], m.predict_proba(frame.iloc[te][cols])[:, 1])

honest_auc = quick_score(FEATS)
print(f"HONEST   AUC (5 features, client-holdout): {honest_auc:.3f}")

### The trap — add ONE label-derived column on purpose

In [ ]:
leaky_auc = quick_score(FEATS + ["h2_over_h1"])      # h2_over_h1 is computed from the label window
print(f"LEAKY    AUC (+ h2_over_h1):               {leaky_auc:.3f}   <- jumps toward 1.0")

`h2_over_h1` is impressions from 16–31 March divided by impressions from 1–15 March — the label's own source. It is not knowable on the 15th, so the score is fake. **Removed below; the honest number is the one that stays.**

In [ ]:
frame = frame.drop(columns=["h2_over_h1"])
assert "h2_over_h1" not in frame.columns
print(f"Kept (honest) AUC: {honest_auc:.3f}   |   leak inflated it by {leaky_auc - honest_auc:+.3f}")

## 4. Data limits
*One named limitation of my slice.*
_Edit this after you see your numbers._ Draft: the label is a **proxy** — a >20% fall in second-half-of-March impressions — measured inside **one month**, so seasonality or a client's own site change can look like "decline", and nothing here shows a refresh would reverse it. The slice also only covers pages with ≥ 100 impressions in 1–15 March, so low-traffic pages are out of scope.

## 5) Self-check
- [ ] Five contract answers written in plain words (section 1)
- [ ] Exactly three verification queries run, outputs visible; availability uses `IS TRUE`
- [ ] Five features, each with a "knowable when" line
- [ ] Leak added, score jump shown, column deleted, honest number kept
- [ ] One named limitation (section 4)
- [ ] No token, client name, or URL pasted anywhere; sealed month `2026-06` untouched
- [ ] Notebook executed top to bottom, then committed under `work/notebooks/`